In [1]:
from crewai import Agent, Task, Crew
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
import os
os.environ["CREWAI_TESTING"] = "true"
from utils import get_openai_api_key, get_exa_api_key
from IPython.display import Markdown
import yaml

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"
# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()
# set the EXA API key
os.environ["EXA_API_KEY"] = get_exa_api_key()

/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/firecrawl/firecrawl.py:78: UserWarning: Field name "json" in "ChangeTrackingData" shadows an attribute in parent "BaseModel"
  class ChangeTrackingData(pydantic.BaseModel):


In [2]:
# create the tool instances
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL")) 
scrape_website_tool = ScrapeWebsiteTool()

# load the configuration file for the agents
with open('config/agents.yaml', 'r') as file:
        agent_config = yaml.safe_load(file)

# create the agents using the configuration
research_planner = Agent(
        config=agent_config['research_planner'],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )
internet_researcher = Agent(
        config=agent_config['internet_researcher'],
        tools=[exa_search_tool, scrape_website_tool],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )
fact_checker = Agent(
        config=agent_config['fact_checker'],
        tools=[exa_search_tool, scrape_website_tool],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )
report_writer = Agent(
        config=agent_config['report_writer'],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )

/var/folders/hf/81yfsz694rx2svmh31g42bb80000gn/T/ipykernel_73736/1076736990.py:2: DeprecationWarning: EXASearchTool is deprecated and will be removed in a future release; use ExaSearchTool instead.
  exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/agent/core.py:358: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/agent/core.py:366: DeprecationWarning: deprecated
  if self.allow_code_execution:
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/agent/core.py:377: DeprecationWarning: deprecated
  if self.reasoning and self.planning_config is None:


In [4]:
import re

# write the custom guardrail function
def write_report_guardrail(output):
    # get the raw output from the TaskOutput object
    try:
        output = output if type(output)==str else output.raw 
    except Exception as e:
        return (False, ("Error retrieving the `raw` argument: "
                        f"\n{str(e)}\n"
                        )
                )
    
    # convert the output to lowercase
    output_lower = output.lower()

    # check that the summary section exists
    if not re.search(r'#+.*summary', output_lower):
        return (False, 
                "The report must include a Summary section with a header like '## Summary'"
                )

    # check that the insights or recommendations sections exist
    if not re.search(r'#+.*insights|#+.*recommendations', output_lower):
        return (False, 
                "The report must include an Insights section with a header like '## Insights'"
                )

    ### START CODE HERE ###

    # check that the citations (or references) section exists
    if not re.search(r'#+.*citations|#+.*references', output_lower):
        return (False,
                "The report must include a Citations (or References) section with a header like '## Citations'"
                )

    ### END CODE HERE ###
    return (True, output)


In [5]:
test_report_pass = """
# Report title

## Executive Summary
This is a summary.

## Insights
These are the insights.

## Citations
1. Citation 1
2. Citation 2
"""

write_report_guardrail(test_report_pass)

(True,
 '\n# Report title\n\n## Executive Summary\nThis is a summary.\n\n## Insights\nThese are the insights.\n\n## Citations\n1. Citation 1\n2. Citation 2\n')

In [6]:
test_report_fail = """
# Report title

## Executive Summary
This is a summary.
"""

write_report_guardrail(test_report_fail)

(False,
 "The report must include an Insights section with a header like '## Insights'")

In [8]:
# load the configuration file for the tasks
with open('config/tasks.yaml', 'r') as file:
    task_config = yaml.safe_load(file)

### START CODE HERE ###

# create the tasks using the configuration
create_research_plan = Task(
    config=task_config['create_research_plan'],
    agent=research_planner
)

gather_research_data = Task(
    config=task_config['gather_research_data'],
    agent=internet_researcher,
)

verify_information_quality = Task(
    config=task_config['verify_information_quality'],
    agent=fact_checker,
)

write_final_report = Task(
    config=task_config['write_final_report'],
    agent=report_writer,
    guardrails=[write_report_guardrail],
)

### END CODE HERE ###

In [9]:
def save_file_hook(result):
    """
    Save the final research report to a local markdown file
    """
    try:
        # Get the final report content from the last task output
        if hasattr(result, 'tasks_output') and result.tasks_output:
            report_content = result.tasks_output[-1].raw
        else:
            report_content = str(result)
        
        filename = f"research_report.md"
        
        # Save to file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_content)
        
        print(f"Report successfully saved to: {filename}")
        
    except Exception as e:
        print(f"Error saving report to file: {str(e)}")

In [10]:
# Create the urban planning crew
deep_research_crew = Crew(
    # include all the agents
    agents=[research_planner, 
            internet_researcher, 
            fact_checker, 
            report_writer],
    # include all the tasks in the order to be executed
    tasks=[create_research_plan, 
           gather_research_data, 
           verify_information_quality, 
           write_final_report],

    ### START CODE HERE ###
    
    # add memory to the crew
    memory=True,
    # add the after kickoff hook
    after_kickoff_callbacks=[save_file_hook]

    ### END CODE HERE ###
)

/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/crew.py:635: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(self.function_calling_llm, LLM):
/opt/homebrew/Cellar/python@3.13/3.13.3/Frameworks/Python.framework/Versions/3.13/lib/python3.13/enum.py:695: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x11cfcb970>
  def __call__(cls, value, names=_not_given, *values, module=None, qualname=None, type=None, start=1, boundary=None):


In [11]:
### START CODE HERE ###

# write your query in the "user_query" value
inputs = {
    "user_query": "Evaluate the top five emerging AI tools for automating competitive market analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing firm."
}

### END CODE HERE ###

In [13]:



import asyncio

# Recommended way in notebooks
result = await deep_research_crew.kickoff_async(inputs=inputs)

/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/telemetry/telemetry.py:431: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if getattr(agent, "function_calling_llm", None)
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/telemetry/telemetry.py:438: DeprecationWarning: deprecated
  "allow_code_execution?": getattr(
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/crews/utils.py:362: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  crew.function_calling_llm,
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/crews/utils.py:104: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if not agent

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Break down the research query "Evaluate the top five emerging AI tools for automating competitive        │
│  market analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing   │
│  firm." into specific topics and key questions that need investigation. Create a focused research plan.         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/rich/text.py:144: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x12acf5120>
  def __init__(


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Research Plan: Evaluating the Top Five Emerging AI Tools for Automating Competitive Market Analysis        │
│                                                                                                                 │
│  #### 1. Main Research Topics to Investigate                                                                    │
│  1. **Identification of Top AI Tools**                                                                          │
│  2. **Features of Each Tool**                                                                                   │
│  3. **Limitations and Challenges**                                                                              │
│  4. **Cost Analysis**                                                                                           │
│  5. **Ideal Use Cases for Mid-Sized Marketing Firms**                                                           │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  #### 2. Key Questions for Each Topic                                                                           │
│                                                                                                                 │
│  **1. Identification of Top AI Tools**                                                                          │
│     - What are the current leading AI tools specifically designed for automating competitive market analysis?   │
│     - How is the effectiveness of these tools measured in the context of market analysis?                       │
│     - Are there any emerging tools that are gaining traction and should be considered?                          │
│                                                                                                                 │
│  **2. Features of Each Tool**                                                                                   │
│     - What core features do these tools offer that are relevant to competitive market analysis?                 │
│     - Are there unique or innovative features that differentiate each tool from its competitors?                │
│     - How user-friendly are the interfaces of these tools for marketers and analysts?                           │
│     - Do the tools support customizations or integrations with other marketing software and platforms?          │
│                                                                                                                 │
│  **3. Limitations and Challenges**                                                                              │
│     - What are the common limitations of these tools in the context of data accuracy and comprehensiveness?     │
│     - Are there challenges faced by users in terms of learning curve or effective usage?                        │
│     - What feedback has been provided by users regarding the reliability and performance of these tools?        │
│     - Are there specific data sources or types of analysis that these tools struggle with?                      │
│                                                        

...

/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/memory/types.py:364: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  age_seconds = (datetime.utcnow() - record.created_at).total_seconds()
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/memory/storage/lancedb_storage.py:352: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().isoformat()
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/memory/storage/lancedb_storage.py:352: DeprecationWarning: datetime.datetime.utcno

Tool search_memory executed with result: Found memories:
- (score=0.82) The five emerging AI tools for automating competitive market analysis are Klue, Crayon, Semrush, Similarweb, and Competely.
  categories: AI Tools, Market Analysis, Comp...
Tool search_memory executed with result: Found memories:
- (score=0.75) Klue's effectiveness is limited by its focus on B2B markets and a steep learning curve for new users.
  categories: B2B, user_experience, product_effectiveness
  entitie...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Comprehensive Report: Evaluation of the Top Five Emerging AI Tools for Automating Competitive Market         │
│  Analysis                                                                                                       │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  In the rapidly evolving environment of digital marketing, mid-sized firms require sophisticated tools for      │
│  competitive market analysis to maintain and enhance their market position. This report evaluates five          │
│  emerging AI tools—Klue, Crayon, Semrush, Similarweb, and Competely—focusing on their features, limitations,    │
│  pricing, and ideal use cases for mid-sized marketing firms. By synthesizing comprehensive insights, this       │
│  report aims to provide actionable recommendations for firms to optimize their marketing strategies             │
│  effectively.                                                                                                   │
│                                                                                                                 │
│  ## Detailed Findings                                                                                           │
│                                                                                                                 │
│  ### 1. Identification of Top AI Tools                                                                          │
│                                                                                                                 │
│  #### Selected Tools                                                                                            │
│  The five AI tools identified are as follows:                                                                   │
│                                                                                                                 │
│  1. **Klue**                                                                                                    │
│     - **Purpose:** Competitive enablement and win-loss analysis.                                                │
│     - **Effectiveness Measurement:** Success is gauged by win rates and sales efficiency metrics (Klue).        │
│                                                                                                                 │
│  2. **Crayon**                                                                                                  │
│     - **Purpose:** Website change monitoring and competitive tracking for real-time insights.                   │
│     - **Effectiveness Measurement:** Assessed through the number of actionable insights generated (Rocket).     │
│                                                                                                                 │
│  3. **Semrush**                                                                                                 │
│     - **Purpose:** SEO analytics and keyword strategy insights.                                                 │
│     - **Effectiveness Measurement:** Evaluated based on

Report successfully saved to: research_report.md


/opt/homebrew/Cellar/python@3.13/3.13.3/Frameworks/Python.framework/Versions/3.13/lib/python3.13/threading.py:303: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x12acf7c40>
  def __enter__(self):
/opt/homebrew/Cellar/python@3.13/3.13.3/Frameworks/Python.framework/Versions/3.13/lib/python3.13/threading.py:303: ResourceWarning: unclosed database in <sqlite3.Connection object at 0x12acf49a0>
  def __enter__(self):
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/crewai/memory/encoding_flow.py:379: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/Users/olaoluadisa/Documents/WEBDEV1/webdev/code files/codefiles/llm/llm_venv/lib/python3.13/site-packages/pydantic/main.py:250: DeprecationWarning: datetime.datetime.utcnow() is deprecated 